# CICIDS2017 Intrusion Detection: PySpark Exploration Walkthrough

This notebook demonstrates step-by-step exploration, feature engineering, and model training using PySpark on the CICIDS2017 network flow dataset.

In [ ]:
import sys
from pathlib import Path

# Ensure project src is in Python path
project_dir = Path.cwd().parent
sys.path.append(str(project_dir / "src"))

from config import get_spark_session, PARQUET_DATA_DIR, EDA_DIR
from pyspark.sql import functions as F

spark = get_spark_session("CICIDS2017-Exploration-Notebook")
print("Spark Version:", spark.version)

## 1. Load Processed Parquet Data

In [ ]:
parquet_path = PARQUET_DATA_DIR / "cleaned_flows.parquet"
df = spark.read.parquet(str(parquet_path))
print(f"Total Cleaned Flows: {df.count():,}")
df.printSchema()

## 2. Attack Class Distribution and Imbalance

In [ ]:
class_counts = df.groupBy("Label").count().orderBy(F.desc("count"))
class_counts.show(20, truncate=False)

## 3. Hourly Attack Infiltration Dynamics

In [ ]:
hourly_attack = (
    df.filter(F.col("Label") != "BENIGN")
    .groupBy("day", "hour")
    .count()
    .orderBy("day", "hour")
)
hourly_attack.show(25)

## 4. Top Selected Features from Feature Engineering Pipeline

In [ ]:
import json
feat_path = EDA_DIR / "selected_features.json"
if feat_path.exists():
    with open(feat_path, "r") as f:
        feat_info = json.load(f)
    print("Top Selected Informative Features:")
    for i, feat in enumerate(feat_info.get("top_features", []), 1):
        print(f"{i:2d}. {feat}")

## 5. Model Evaluation Metrics

In [ ]:
import pandas as pd
from config import METRICS_DIR

metrics_csv = METRICS_DIR / "model_comparison.csv"
if metrics_csv.exists():
    metrics_df = pd.read_csv(metrics_csv)
    display(metrics_df)